In [ ]:
import NPI
import pandas as pd
from lesion_response import *
from local_lpcmci import *
from intervention_optimization import *
from recovery_evaluation import *
from hypothesis_lagged_validation import *
from centroid_calculation import *
from stability_analysis import run_n_seed_windows_convergence_check
from stage1_visualization import *
from stage2_visualization import *
from stage3_visualization import *
from stage4_visualization import *
from stage4_network_visualization import (
    plot_stage4_directed_causal_intervention_graph_split,
    extract_policy_lesion_paths,
    build_policy_lesion_path_subgraph,
    plot_stage4_policy_lesion_path_graph,
    plot_stage4_policy_lesion_recovery_path_graph_pair
)
from simulated_var_validation import (
    run_simulated_var_validation_experiment,
    summarize_simulated_var_results,
    judge_simulated_var_validation,
    build_simulated_var_two_layer_tables,
    build_screen_ratio_comparison_table
)
from stability_analysis import (
    run_topk_threshold_sensitivity,
    summarize_topk_threshold_sensitivity,
    plot_topk_sensitivity_metrics,
    plot_best_node_frequency,
)
from Parametric_grid_search import (
    BASE_CONFIG,
    OUTPUT_DIR,
    run_stagewise_grid_search_with_constraints,
    run_full_pipeline_with_config,
    summarize_final_pipeline_result,
)

In [ ]:
def flat_without_diagonal(matrix):

    "Flatten the matrix without including the diagonal"
    
    n = matrix.shape[0]
    flattened = []
    for i in range(n):
        for j in list(range(i)) + list(range(i + 1, n)): #list拼接 可以直接用i!=j
            flattened.append(matrix[i][j])
    
    return np.array(flattened)

In [ ]:
NPI.set_seed(42)
batch_size = 50
train_set_proportion = 0.8
ROI_num = 360
using_steps = 3
pred_steps = 1     

num_epochs = 100
lr = 1e-3
l2 = 5e-5
patience = 10
min_delta = 1e-5
dropout = 0.2
noise_std = 0.0

In [ ]:
#信号处理 分割成输入数据和输出数据,数据条数共1200条 脑区数量360
signals = []
inputs = []
targets = []
for file in ['./data/raw/real_fMRI_data/102008_REST1_LR.txt',
             './data/raw/real_fMRI_data/102008_REST1_RL.txt',
             './data/raw/real_fMRI_data/102008_REST2_LR.txt',
             './data/raw/real_fMRI_data/102008_REST2_RL.txt']:
    data = np.loadtxt(file)[30:, :360].astype(np.float32)
    signals.append(data)
    input, target = NPI.multi2one(data, input_steps=using_steps, pred_steps=pred_steps)
    inputs.append(input)
    targets.append(target)
signals = np.vstack(signals).astype(np.float32)
inputs = np.vstack(inputs).astype(np.float32)
targets = np.vstack(targets).astype(np.float32)

In [ ]:
#neworderlr 给出了脑区的新排列方式； neworder 表示旧脑区在新顺序中的位置映射； oldorder 是其逆映射，用于从新顺序恢复到原顺序。
neworderlr = np.loadtxt('./data/derived/Whole-brain_EC/MNI-MMP1/neworderlr.txt')
print(neworderlr)
print(neworderlr.shape)
#旧脑区 i 在新顺序中的位置 neworder[i] = 新顺序中的位置
#np.where(neworderlr == 0) → (array([1]),) np.where(neworderlr == i)[0][0] (array([1]),)[0] → array([1]) array([1])[0] → 1
neworder = np.array([np.where(neworderlr == i)[0][0] for i in range(len(neworderlr))])
#oldorder[i] 代表i号脑区在最开始的脑区的位置
oldorder = np.array([np.where(neworder == i)[0][0] for i in range(len(neworder))])
# neworder 是 old → new
# oldorder 是 new → old
# 它们是 互逆置换。
# 因为在不同步骤：fMRI 时间序列 FC / EC 矩阵 DTI 结构矩阵 ROI 的顺序不统一 👉 必须能 来回切换 ROI 顺序
# 脑区重排序让不同来源的数据在同一语义坐标系里对齐 脑区重排的目的：是让“同一个索引，在所有数据里都指向同一个脑区”。


In [ ]:
ANN = NPI.ANN_MLP(input_dim = using_steps * ROI_num, hidden_dim = 2 * ROI_num, latent_dim = int(0.8 * ROI_num), output_dim = ROI_num,pred_steps=pred_steps, dropout=dropout)
# ANN = NPI.ANN_CNN(in_channels = ROI_num, hidden_channels = 3 * ROI_num, out_channels = ROI_num, data_length = using_steps,pred_steps=pred_steps)
# ANN = NPI.ANN_RNN(input_dim = ROI_num, hidden_dim = int(2.5 * ROI_num), latent_dim = int(2.5 * ROI_num), output_dim = ROI_num, data_length = using_steps,pred_steps=pred_steps)
# ANN = NPI.ANN_VAR(input_dim = using_steps * ROI_num, output_dim = ROI_num,pred_steps=pred_steps)

ANN, history = NPI.train_NN(
    ANN,
    inputs,
    targets,
    pred_steps=pred_steps,
    batch_size=batch_size,
    train_set_proportion=train_set_proportion,
    num_epochs=num_epochs,
    lr=lr,
    l2=l2,
    patience=patience,
    noise_std=noise_std
)

print("\nBest epoch:", history["best_epoch"])
print("Best test loss:", history["best_test_loss"])

In [ ]:
fig, ax = plt.subplots(figsize=(6.2, 4.4))

ax.plot(
    history["epoch"],
    history["train_noisy_loss"],
    label="Train noisy loss",
    linewidth=2.0,
    alpha=0.85
)

ax.plot(
    history["epoch"],
    history["train_clean_loss"],
    label="Train clean loss",
    linewidth=2.2
)

ax.plot(
    history["epoch"],
    history["test_loss"],
    label="Test loss",
    linewidth=2.2
)

best_epoch = history["best_epoch"]
best_test_loss = history["best_test_loss"]

ax.axvline(
    best_epoch,
    linestyle="--",
    linewidth=1.3,
    alpha=0.8,
    label=f"Best epoch = {best_epoch}"
)

ax.scatter(
    [best_epoch],
    [best_test_loss],
    s=45,
    zorder=5
)

ax.set_xlabel("Epoch")
ax.set_ylabel("MSE loss")
ax.set_title("Training and Testing Loss Curves")
ax.legend(frameon=False)
ax.grid(alpha=0.25, linestyle="--")

plt.tight_layout()
plt.show()

In [ ]:
#empirical_FC 是一个 ROI×ROI 的功能连接矩阵，其中每个元素表示对应两个脑区时间序列之间（整个时间维度上 计算皮尔逊相关系数）的皮尔逊相关系数。矩阵中的每一个点对应一条脑区之间的连接，而不是某一时刻的信号值。
#每一对脑区的相关系数 是用 这 12000 个点一起算出来的
# empirical_FC = NPI.corrcoef(signals) #从真实 fMRI 时间序列，算 ROI–ROI 的相关矩阵  即ROI i 和 ROI j进行比较 看看他们在整个时间轴上是不是“同步变化”
# model_FC = NPI.model_FC(ANN, node_num = ROI_num, steps = using_steps)#使用模型去模拟时间序列 然后算出FC
# print(empirical_FC.shape)
# 6. Held-out predictive FC + affine calibration + GSR
def fit_roi_affine(pred_train, true_train, eps=1e-8):
    """
    pred_train, true_train: shape (n_samples, ROI_num)
    拟合每个 ROI 的线性校正：y = a*x + b
    """
    mu_x = pred_train.mean(axis=0)
    mu_y = true_train.mean(axis=0)

    var_x = pred_train.var(axis=0)
    cov_xy = ((pred_train - mu_x) * (true_train - mu_y)).mean(axis=0)

    a = cov_xy / (var_x + eps)
    b = mu_y - a * mu_x
    return a, b
def apply_roi_affine(pred, a, b):
    """
    pred: shape (n_samples, ROI_num)
    """
    return pred * a[None, :] + b[None, :]


def remove_global_signal(X):
    """
    X shape: (n_samples, ROI_num)
    每个时间点减去该时刻全脑平均值
    """
    return X - X.mean(axis=1, keepdims=True)
# =========================================================
# Held-out predictive FC + affine calibration + GSR
# 一步预测版本
# =========================================================

train_X, train_Y, test_X, test_Y = NPI.split_train_test(
    inputs, targets, train_set_proportion=train_set_proportion
)

pred_train_all = NPI.predict_array(ANN, train_X, batch_size=256)
true_train_all = train_Y.reshape(-1, pred_steps, ROI_num)

pred_test_all = NPI.predict_array(ANN, test_X, batch_size=256)
true_test_all = test_Y.reshape(-1, pred_steps, ROI_num)

alpha = 0.8

# 一步预测只有 step_idx = 0
step_idx = 0

pred_train_k = pred_train_all[:, step_idx, :]
true_train_k = true_train_all[:, step_idx, :]

# 训练集拟合 ROI-wise affine calibration
a, b = fit_roi_affine(pred_train_k, true_train_k)

a_shrink = 1.0 + alpha * (a - 1.0)
b_shrink = alpha * b

# 测试集预测
pred_test_k = pred_test_all[:, step_idx, :]
true_test_k = true_test_all[:, step_idx, :]

# affine calibration
pred_test_k_cal = apply_roi_affine(pred_test_k, a_shrink, b_shrink)

# global signal removal
pred_test_k_fc = remove_global_signal(pred_test_k_cal)
true_test_k_fc = remove_global_signal(true_test_k)

# FC
model_FC_t1 = NPI.corrcoef(pred_test_k_fc)
empirical_FC_t1 = NPI.corrcoef(true_test_k_fc)

# 如果你后面代码习惯用 model_FC / empirical_FC，也可以这样赋值
model_FC = model_FC_t1
empirical_FC = empirical_FC_t1

In [ ]:
# NPI_EC = NPI.model_EC(ANN, inputs, targets, pert_strength = 1.0)
# np.fill_diagonal(NPI_EC, 0)
NPI_EC = NPI.model_EC(
    ANN,
    inputs,
    targets,
    pert_strength=1.0,
    pred_steps=pred_steps,
    step_idx=0,
    batch_size=256
)
np.fill_diagonal(NPI_EC, 0)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize = (12, 6))
sns.heatmap(empirical_FC[neworder].T[neworder].T, ax = ax1, vmin = -1.0, vmax = 1.0, cmap = 'RdBu_r', cbar = False, square = True, xticklabels = False, yticklabels = False)
sns.heatmap(model_FC[neworder].T[neworder].T, ax = ax2, vmin = -1.0, vmax = 1.0, cmap = 'RdBu_r', cbar = False, square = True, xticklabels = False, yticklabels = False)
ax1.set_title('Empirical FC'); ax2.set_title('Model FC')
plt.show()

In [ ]:
# r_value, p_value = pearsonr(flat_without_diagonal(model_FC), flat_without_diagonal(empirical_FC))
#
# plt.figure(figsize = (4.8, 4.8))
# plt.scatter(flat_without_diagonal(model_FC), flat_without_diagonal(empirical_FC))
#
# plt.xlim(-0.7, 1.1); plt.xticks([-0.5, 0.0, 1.0]); plt.xlabel('model FC')
# plt.ylim(-0.7, 1.1); plt.yticks([-0.5, 0.0, 1.0]); plt.ylabel('empirical FC')
# plt.text(-0.6, 0.9, 'r = {:.2f}, p = {:.0e}'.format(r_value, p_value))
#
# ax = plt.gca()
# ax.spines['right'].set_visible(False); ax.spines['top'].set_visible(False)
#
# plt.show()
#散点图每个点是一个脑区对的 FC 值 (model_FC(i,j), empirical_FC(i,j))。
# 散点图用于评估模型连接与经验连接之间的一致性。横轴是模型生成的连接强度，纵轴是经验数据中的连接强度。每一个点代表一对脑区之间的连接边，在绘制前我们去除了对角线并将矩阵展平成一维向量。从结果可以看到二者呈现出明显的正相关关系，Pearson 相关系数为 0.69，说明模型在整体连接结构层面上能够较好地重现经验脑网络特征。
#r 值表示模型生成的功能连接与经验功能连接在整体结构上的线性相关程度，越接近 1 表示越一致；p 值表示该相关性是否显著，越小说明相关性越不可能是随机出现。
# 计算 r 和 p
#r 值越接近 1 → 散点沿拟合线线性分布 → 拟合越好 r 值接近 0 → 散点随机分布 → 拟合差
#散点沿红色拟合线均匀排列 → 拟合好 散点围绕 45° 对角线 → 模型与真实数据一致 r 越大、p 越小 → 拟合越可信
r_value, p_value = pearsonr(flat_without_diagonal(model_FC), flat_without_diagonal(empirical_FC))

# 设置画布
plt.figure(figsize=(4.8, 4.8))

# 散点图 + 拟合红线
sns.regplot(
    x=flat_without_diagonal(model_FC),
    y=flat_without_diagonal(empirical_FC),
    scatter_kws={'s': 10, 'alpha': 0.6},  # 点小且半透明
    line_kws={'color': 'red'}             # 拟合线红色
)

# 坐标轴范围与刻度
plt.xlim(-0.7, 1.1)
plt.xticks([-0.5, 0.0, 1.0])
plt.xlabel('model FC')

plt.ylim(-0.7, 1.1)
plt.yticks([-0.5, 0.0, 1.0])
plt.ylabel('empirical FC')

# 在图上显示 r 和 p
plt.text(-0.6, 0.9, f'r = {r_value:.2f}, p = {p_value:.0e}')

# 去掉右边框和上边框
ax = plt.gca()
ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)

plt.title('Model vs Empirical FC', fontsize=12)
plt.show()

In [ ]:
fig = plt.figure(figsize = (10, 10))
sns.heatmap(NPI_EC[neworder].T[neworder].T[:180, :180], vmin = -0.1, vmax = 0.1, cmap = 'RdBu_r', cbar = True, square = True, xticklabels = False, yticklabels = False)
plt.show()
fig = plt.figure(figsize = (10, 10))
sns.heatmap(
    NPI_EC[neworder].T[neworder].T[180:360, 180:360],
    vmin = -0.1, vmax = 0.1,
    cmap = 'RdBu_r',
    cbar = True, square = True,
    xticklabels = False, yticklabels = False
)
plt.show()

#新的脑区顺序下，前180个脑区两两之间的有效连接强度。颜色表示的增强还是抑制，以及强弱程度。蓝色表示抑制，红色表示增强
#展示了180个脑区 只看左半球内部的 EC 结构 整个大脑有360个脑区


stage1 lesion response

In [ ]:
lesion_idx = 50
label_file = "./data/derived/Whole-brain_EC/MNI-MMP1/labels_MNI-MMP1.txt"
roi_names = load_mmp_labels_cortical_only(label_file)
# 运行分阶段网格搜索
search_res = run_stagewise_grid_search_with_constraints(
    base_config=BASE_CONFIG,
    model=ANN,
    inputs=inputs,
    signals=signals,
    roi_num=ROI_num,
    using_steps=using_steps,
    lesion_idx=lesion_idx,
    empirical_FC=empirical_FC,
    roi_names=roi_names,
    top_k=20,
    output_dir=OUTPUT_DIR,
)

BEST_CONFIG = search_res["BEST_CONFIG"]
import json
# 保存
with open("./results/grid_search/BEST_CONFIG.json", "w", encoding="utf-8") as f:
    json.dump(BEST_CONFIG, f, indent=2, ensure_ascii=False)
# 下次读取
# with open("./results/grid_search/BEST_CONFIG.json", "r", encoding="utf-8") as f:
#     BEST_CONFIG = json.load(f)

In [ ]:
stage1_res = compute_lesion_response(
    model=ANN,
    inputs=inputs,
    roi_num=ROI_num,
    roi_names=roi_names,
    using_steps=using_steps,
    lesion_idx=lesion_idx,
    
    horizon=BEST_CONFIG["horizon"],
    n_seed_windows=BEST_CONFIG["n_seed_windows"],
    lesion_strength=BEST_CONFIG["lesion_strength"],
    lesion_mode=BEST_CONFIG["lesion_mode"],
    random_state=BEST_CONFIG["random_state"],
    batch_size=BEST_CONFIG["batch_size"],
)
stage1_res.keys()
stage1_res["lesion_response_df"].head(20)


In [ ]:
diag2 = plot_stage1_response_and_stability(
    stage1_res,
    figsize=(13.5, 8.0),
    save_path="./results/stage1_response_and_stability_combo.png",
)

display(diag2["phase_df"])
display(diag2["summary_df"])
diag2["diagnostics"]

In [ ]:
plot_top_response_bar(stage1_res, top_k=20,roi_names=roi_names,show_roi_id=False)

In [ ]:
# 画 Stage 1 lesion response brain map:三视角 brain map
lesion_idx = 50
# 把brainmap_df读进来
#brainmap_df=r"./data/derived/MNI-MMP_centroid/stage1_brainmap_df.csv"
csv_file_path = r"./data/derived/MNI-MMP_centroid/stage1_brainmap_df.csv"
brainmap_df = pd.read_csv(csv_file_path)
fig, axes = plot_stage1_lesion_response_brainmap(
    brainmap_df=brainmap_df,
    value_col="response_signed",    # 也可看response_abs，换cmap="Reds"
    lesion_idx=lesion_idx,
    top_k_label=5,
    figsize=(15.5, 4.8),
    cmap="RdBu_r",
    annotate_only_xz=True,
    save_path="./results/stage1_brainmap_signed_numbered.png",
)

In [ ]:
fig, ax = plot_stage1_centroid_brainmap_paper(
    brainmap_df=brainmap_df,
    value_col="response_abs",  #也可response_signed
    lesion_idx=lesion_idx,
    top_k_label=5,
    cmap=None,
    title="Stage 1 lesion-response map",
    save_path="./results/stage1_xz_response_abs_numbered.png"
)

In [ ]:
dlabel_path = r"./data/derived/MNI-MMP_centroid/HCP_MMP1.0_Glasser.32k_fs_LR.dlabel.nii"
left_surf_path = r"./data/derived/MNI-MMP_centroid/S1200.L.midthickness_MSMAll.32k_fs_LR.surf.gii"
right_surf_path = r"./data/derived/MNI-MMP_centroid/S1200.R.midthickness_MSMAll.32k_fs_LR.surf.gii"
#mmp360_centroids_aligned=r"./data/derived/MNI-MMP_centroid/mmp360_centroids_aligned_to_model.csv"
csv_file_path = r"./data/derived/MNI-MMP_centroid/mmp360_centroids_aligned_to_model.csv"
mmp360_centroids_aligned = pd.read_csv(csv_file_path)
lesion_df = stage1_res["lesion_response_df"].copy()
lesion_idx = 50
fig_surface, axes_surface = plot_stage1_surface_brainmap(
    dlabel_path=dlabel_path,
    left_surf_path=left_surf_path,
    right_surf_path=right_surf_path,
    lesion_df=lesion_df,
    aligned_centroid_df=mmp360_centroids_aligned,
    lesion_idx=lesion_idx,
    value_col="response_abs",
    cmap="Reds",
    save_path="./results/stage1_surface_response.png",
    suptitle="Stage 1 lesion response surface map"
)

In [ ]:
node_subset = get_top_response_node_subset(
    brainmap_df=brainmap_df,
    value_col="response_abs",
    lesion_idx=lesion_idx,
    top_n=30
)

edge_df_top_response = build_high_fc_edge_df(
    fc_matrix=empirical_FC_t1,
    node_subset=node_subset,
    edge_mode="top_k_per_node",
    top_k_per_node=2
)

fig, axes, fc_subnet_info = plot_stage1_fc_subnetwork_matrix(
    brainmap_df=brainmap_df,
    fc_matrix=empirical_FC_t1,
    lesion_idx=lesion_idx,
    value_col="response_abs",
    roi_names=roi_names,
    node_subset=node_subset,
    top_k=20,
    lesion_first=True,
    title="Stage 1 lesion-responsive FC subnetwork",
    save_path="./results/stage1_fc_subnetwork_matrix_paper.png",
)

stage 2 local lpcmci

In [ ]:
# whole-brain lagged candidate screening
# 全脑范围内计算 Xi(t-1) → Xj(t) 的滞后关联强度；
# 结合 Stage 1 病灶响应先验；
# 筛出病灶相关候选节点；
# 暂时不让 FC 参与筛选，保证 FC 只在后面用于图压缩。
screen_res = run_lagged_candidate_screening(
    signals=signals,
    lesion_idx=lesion_idx,
    roi_names=roi_names,
    lesion_response_prior=stage1_res,     # Stage 1 dynamic abnormality prior
    empirical_FC=None,                    # keep FC out of screening; FC is used only in pruning
    
    tau=BEST_CONFIG["tau"],
    max_candidate_nodes=BEST_CONFIG["max_candidate_nodes"],
    min_candidate_nodes=BEST_CONFIG["min_candidate_nodes"],
    top_k_direct_lagged=BEST_CONFIG["top_k_direct_lagged"],
    top_k_response=BEST_CONFIG["top_k_response"],
    top_k_fc=BEST_CONFIG["top_k_fc"],

    w_lagged=BEST_CONFIG["w_lagged"],
    w_response=BEST_CONFIG["w_response"],
    w_fc=BEST_CONFIG["w_fc"],

    standardize=BEST_CONFIG["standardize"],
)

print(screen_res["info"])
print(screen_res["candidate_nodes"][:20])
screen_res["node_score_df"].head(30)

In [ ]:
plot_screening_node_scores(
    node_score_df=screen_res["node_score_df"],
    top_k=20,
    roi_names=roi_names,
    lesion_idx=lesion_idx,
    stage1_res=stage1_res,
    x_rotation=60
)

In [ ]:
# LPCMCI on screened candidate subnetwork
# node_map 的作用是：LPCMCI 在局部子网上跑，内部编号是 0,1,2,...
# node_map 负责把局部编号映射回原始全脑 ROI 编号。
local_lpcmci_res, node_map = run_candidate_subnetwork_lpcmci(
    signals=signals,
    candidate_nodes=screen_res["candidate_nodes"],
    tau_min=BEST_CONFIG["tau_min"],
    tau_max=BEST_CONFIG["tau_max"],
    pc_alpha=BEST_CONFIG["pc_alpha"],
    verbosity=0,
    standardize=BEST_CONFIG["standardize"],
    max_cond_px=1,
    max_p_global=2,
    max_p_non_ancestral=2,
    max_q_global=5,
    max_pds_set=8,
)
# 把 LPCMCI 结果转成全脑 ROI 边表，不使用同期边。
local_edge_df = lpcmci_to_edge_df(
    local_lpcmci_res,
    node_map=node_map,
    include_contemp=False,
    lag_only=True,
    alpha_level=BEST_CONFIG["alpha_level"],
)

print("local LPCMCI lagged edges:", len(local_edge_df))
local_edge_df.head(20)

In [ ]:
# lesion-centered first/second-order LPCMCI neighborhood：提取病灶中心一阶 / 二阶邻域
#先取病灶节点的一阶因果邻居；
# 如果一阶邻居太少，扩展到二阶；
# 如果节点太多，按 LPCMCI 强度压缩节点。
lesion_nodes, lesion_sub_edges, subnet_info = extract_lesion_neighborhood(
    edge_df=local_edge_df,
    lesion_idx=lesion_idx,
    min_nodes=BEST_CONFIG["min_lesion_sub_nodes"],
    max_nodes=BEST_CONFIG["max_lesion_sub_nodes"],
    direction="both",
    expand_to_second_order=BEST_CONFIG["expand_to_second_order"],
)

display(subnet_info)
lesion_sub_edges.head(20)

In [ ]:
# FC约束删边，规则：
#直接连接 lesion 的边优先保留；
# p 值显著且 LPCMCI 强度高的边优先保留；
# FC 只用于边太多时的压缩；
# FC 不用于否定强因果边。
lesion_core_edges, prune_info = prune_edges_by_lpcmci_fc_constraints(
    sub_edges=lesion_sub_edges,
    empirical_FC=empirical_FC,
    lesion_idx=lesion_idx,
    max_edges=BEST_CONFIG["max_edges"],
    alpha_level=BEST_CONFIG["prune_alpha_level"],
    strong_quantile=BEST_CONFIG["strong_quantile"],
    causal_weight=BEST_CONFIG["causal_weight"],
    fc_weight=BEST_CONFIG["fc_weight"],
    max_direct_lesion_edges=BEST_CONFIG["max_direct_lesion_edges"],
)

display(prune_info)
lesion_core_edges.head(20)

In [ ]:
# 生成干预候选节点
candidate_df = build_intervention_candidates_from_core(
    lesion_core_edges,
    lesion_idx=lesion_idx,
    exclude_lesion=True,
)

candidate_df.head(20)

In [ ]:
# 证明方法合理性
overlap_info = compute_stage1_screening_overlap(
    stage1_res=stage1_res,
    screen_res=screen_res,
    stage1_top_k=30,
    screening_top_k=30,
)

overlap_info

In [ ]:
# overlap图
plot_stage2_screening_overlap(
    overlap_info=overlap_info,
    roi_names=roi_names,
    title="Stage 1 response and lagged-screening overlap",
    max_overlap_show=8,
    save_path="./results/stage1_stage2_overlap.png"
)

In [ ]:
# 直接 lesion 子图
direct_lesion_edges = lesion_core_edges[
    (lesion_core_edges["src"].astype(int) == int(lesion_idx))
    | (lesion_core_edges["dst"].astype(int) == int(lesion_idx))
].copy()

fig_core, axes_core, core_info = plot_lpcmci_lesion_core_graph(
    lesion_core_edges=direct_lesion_edges,
    lesion_idx=lesion_idx,
    candidate_df=candidate_df,
    stage1_res=stage1_res,
    roi_names=roi_names,
    node_score_source="response_abs",

    max_edges=30,
    top_score_nodes=12,
    top_label_k=8,

    edge_alpha=0.42,
    edge_width_scale=2.2,

    node_size_min=220,
    node_size_scale=440,
    lesion_node_size=980,

    title="Stage 2 direct lesion-connected LPCMCI core graph",
    save_path="./results/stage2_direct_lesion_core_graph_split.png",
)

display(core_info["edge_df_plotted"].head())

In [ ]:
# 直接lesion子图
direct_lesion_edges = lesion_core_edges[
    (lesion_core_edges["src"].astype(int) == int(lesion_idx))
    | (lesion_core_edges["dst"].astype(int) == int(lesion_idx))
].copy()
fig_resp, ax_resp, fig_edge, ax_edge, graph_info = plot_lpcmci_lesion_core_graph_split(
    lesion_core_edges=direct_lesion_edges,
    lesion_idx=lesion_idx,
    candidate_df=candidate_df,
    stage1_res=stage1_res,
    roi_names=roi_names,
    node_score_source="response_abs",

    max_edges=30,
    top_score_nodes=12,
    top_label_k=8,

    edge_alpha=0.45,
    edge_width_scale=1.8,

    node_size_min=240,
    node_size_scale=480,
    lesion_node_size=1050,

    title_prefix="Stage 2 direct lesion-connected LPCMCI core graph",
    save_prefix="./results/stage2_direct_lesion_core_graph",
)

In [ ]:
# 全局 Top-K strongest core edges
fig, ax, top_edge_df = plot_stage2_top_core_edges(
    lesion_core_edges=lesion_core_edges,
    top_k=15,
    roi_names=roi_names,
    lesion_idx=lesion_idx,
    plot_type="lollipop",
    direct_lesion_only=False,   # 也可看lesion 直接相连的 Top-K core edges
    title="Top 15 strongest LPCMCI core edges",
    save_path="./results/stage2_top15_core_edges_lollipop.png",
)

In [ ]:
#邻接矩阵热图
fig, ax, heatmap_info = plot_lpcmci_adjacency_heatmap(
    lesion_core_edges=lesion_core_edges,   # 一定传 core edges，不要传 local edges
    lesion_idx=lesion_idx,
    candidate_df=candidate_df,
    stage1_res=stage1_res,
    roi_names=roi_names,
    node_score_source="response_abs",
    value_col="val",

    max_nodes=40,
    figsize=(10.5, 8.4),

    title="Stage 2 LPCMCI core-edge adjacency heatmap",
    grid_linewidth=0.04,

    save_path="./results/stage2_lpcmci_core_adjacency_heatmap.png"
)

In [ ]:
# 图稀疏化过程柱状图
stage2_summary = {
    "n_screen_nodes": len(screen_res["candidate_nodes"]),
    "n_local_edges": len(local_edge_df),
    "n_lesion_sub_edges": len(lesion_sub_edges),
    "n_core_edges": len(lesion_core_edges),
}

fig, ax, sparsify_df = plot_stage2_sparsification_summary(
    summary=stage2_summary,
    title="Stage 2 sparsification process",
    save_path="./results/stage2_sparsification_summary.png",
)
sparsify_df

In [ ]:
# 图稀疏化过程流程图
fig, ax = plot_stage2_sparsification_flowchart(
    summary=stage2_summary,
    title="Stage 2 graph sparsification",
    save_path="./results/stage2_sparsification_flowchart.png",
)

stage 3 intervention

In [ ]:
# 单节点 + greedy 多节点最优干预
candidate_nodes = candidate_df["node"].head(20).astype(int).drop_duplicates().tolist()
candidate_nodes = [
    n for n in candidate_nodes
    if int(n) != int(lesion_idx)
]
ROLLOUT_HORIZON = stage1_res["healthy_traj"].shape[1]
stage3_res = run_single_node_search(
    model=ANN,
    candidate_nodes=candidate_nodes,
    roi_num=ROI_num,
    using_steps=using_steps,
    lesion_idx=lesion_idx,

    seed_inputs=stage1_res["seed_inputs"],
    healthy_traj=stage1_res["healthy_traj"],
    lesion_traj=stage1_res["lesion_traj"],

    horizon=BEST_CONFIG["horizon"],
    lesion_strength=BEST_CONFIG["lesion_strength"],
    lesion_mode=BEST_CONFIG["lesion_mode"],

    signs=BEST_CONFIG["signs"],
    strengths=BEST_CONFIG["strengths"],
    schedules=BEST_CONFIG["schedules"],

    alpha_traj=BEST_CONFIG["alpha_traj"],
    beta_fc=BEST_CONFIG["beta_fc"],
    lambda_node=BEST_CONFIG["lambda_node"],
    lambda_strength=BEST_CONFIG["lambda_strength"],
    lambda_temporal=BEST_CONFIG["lambda_temporal"],

    batch_size=BEST_CONFIG["batch_size"],

    run_greedy=BEST_CONFIG["run_greedy"],
    max_interventions=BEST_CONFIG["max_interventions"],
    top_m_per_step=BEST_CONFIG["top_m_per_step"],
    min_marginal_gain=BEST_CONFIG["min_marginal_gain"],
    force_accept_first=BEST_CONFIG["force_accept_first"],
)

display(stage3_res["single_df"].head())
display(stage3_res["greedy_history"])
display(stage3_res["greedy_trial_df"].head())

print("Best single policy:")
print(stage3_res["best_single_policy"])

print("Best final policy:")
print(stage3_res["best_policy"])
print("Best final objective:")
print(stage3_res["best_objective"])
# 后面需要扩展时修改
# schedules=("constant","early_pulse","middle_pulse","late_pulse","ramp_up","ramp_decay","exponential_decay")

In [ ]:
single_df = attach_candidate_scores(
    single_df=stage3_res["single_df"],
    candidate_df=candidate_df,
    roi_names=roi_names if "roi_names" in globals() else None,
)
single_df.head(10)
stage3_res["single_df_aug"] = attach_candidate_scores(
    single_df=stage3_res["single_df"],
    candidate_df=candidate_df,
    roi_names=roi_names if "roi_names" in globals() else None,
)

display(stage3_res["single_df_aug"].head(10))

In [ ]:
# 单节点objective ranking bar
fig, ax = plot_stage3_objective_ranking_bar(
    single_df=stage3_res["single_df"],
    top_k=10,
    roi_names=roi_names,
    mode="grouped",  # objective_only更间接一点但解释力不如grouped
    show_cost=True,
    cost_mode="inset",
    x_rotation=45,
    mark_best=True,
    legend_outside=True,
    figsize=(10.8, 4.8),
    save_path="./results/stage3_top10_candidate_intervention_nodes_with_cost.png"
)

In [ ]:
# best-node policy grid heatmap
# 先取最优节点
best_node = int(single_df.iloc[0]["node"])
best_grid = stage3_res["all_grid"][best_node]
plot_stage3_bestnode_policy_grid_heatmap(
    grid_df=best_grid,
    node=best_node,
    metric_col="objective",   # 想画 traj_recovery / fc_recovery改就行
    save_path="./results/stage3_bestnode_policy_grid_heatmap.png",
)

In [ ]:
# 画 best node 的 grid heatmap
#objective heatmap
fig, axes = plot_stage3_best_node_grid_heatmap(
    grid_df=stage3_res["all_grid"][best_node],
    metric="objective",
    title=f"Best-node policy grid heatmap: node {best_node} (objective)",
    figsize=(10.8, 4.8),
    annotate=True,
    cmap=None,
    save_path="./results/stage3_best_node_grid_objective.png"
)

In [ ]:
# traj_recovery heatmap
plot_stage3_best_node_grid_heatmap(
    grid_df=stage3_res["all_grid"][best_node],
    metric="traj_recovery",
    title=f"Best-node policy grid heatmap: node {best_node} (traj_recovery)",
    cmap=None,
    save_path="./results/stage3_best_node_grid_traj_recovery.png"
)

In [ ]:
# fc_recovery heatmap
plot_stage3_best_node_grid_heatmap(
    grid_df=stage3_res["all_grid"][best_node],
    metric="fc_recovery",
    title=f"Best-node policy grid heatmap: node {best_node} (fc_recovery)",
    cmap=None,
    save_path="./results/stage3_best_node_grid_fc_recovery.png"
)

In [ ]:
# Top-K candidate objective decomposition 对比图
plot_stage3_topk_objective_decomposition(
    single_df=single_df,
    top_k=8,
    alpha_traj=1.0,
    beta_fc=0.8,
    lambda_strength=0.05,
    lambda_temporal=0.03,
    label_col="roi_name",
    x_rotation=45,
    legend_outside=True,
    title="Top 8 candidate objective decomposition",
    save_path="./results/stage3_top8_objective_decomposition_roi_name.png",
)

In [ ]:
# 干预策略搜索路径
fig, ax, search_path_summary = plot_stage3_intervention_search_path_graph(
    stage3_res=stage3_res,
    roi_names=roi_names if "roi_names" in globals() else None,
    top_k_show=10,
    figsize=(13.5, 5.8),
    title="Stage 3 intervention search path",
    save_path="./results/stage3_intervention_search_path.png",
)

print(search_path_summary)

In [ ]:
# 贪心搜索过程
fig, ax, greedy_summary = plot_stage3_greedy_policy_construction_path(
    greedy_history=stage3_res["greedy_history"],
    roi_names=roi_names if "roi_names" in globals() else None,
    figsize=(12.5, 5.2),
    title="Stage 3 greedy policy construction path",
    save_path="./results/stage3_greedy_policy_construction_path.png",
)

print(greedy_summary)

In [ ]:
# 搜索过程依据
fig3, ax3, heatmap_info = plot_stage3_candidate_marginal_gain_map(
    greedy_trial_df=stage3_res["greedy_trial_df"],
    roi_names=roi_names if "roi_names" in globals() else None,
    mode="heatmap",   # 也可换ranking版
    top_show_nodes=18,
    title_prefix="Stage 3 candidate marginal gain",
    save_path="./results/stage3_candidate_marginal_gain_heatmap.png",
)

display(heatmap_info["heatmap_df"])

In [ ]:
# 核心定量图
fig1, axes1, combo_summary = plot_stage3_objective_marginal_gain_combo(
    greedy_history=stage3_res["greedy_history"],
    roi_names=roi_names if "roi_names" in globals() else None,
    baseline_objective=0.0,
    title="Stage 3 objective and marginal gain",
    save_path="./results/stage3_objective_marginal_gain_combo.png",
)

print(combo_summary)

In [ ]:
# objective分解过程
fig2, ax2, decomp_df = plot_stage3_objective_decomposition_stackedbar(
    greedy_history=stage3_res["greedy_history"],
    roi_names=roi_names if "roi_names" in globals() else None,
    alpha_traj=1.0,
    beta_fc=0.8,
    lambda_node=0.01,
    lambda_strength=0.05,
    lambda_temporal=0.03,
    title="Stage 3 objective decomposition",
    save_path="./results/stage3_objective_decomposition_stackedbar.png",
)

display(decomp_df)

stage 4 recovery

In [ ]:
# =========================================================
# Stage 4: use final policy from Stage 3
# =========================================================
best_policy = stage3_res["best_policy"]
best_node = int(best_policy[0]["node"])   # primary policy node, usually greedy step 1

if "best_policy_traj" in stage3_res and stage3_res["best_policy_traj"] is not None:
    best_policy_traj = stage3_res["best_policy_traj"]
elif "best_multi_policy_traj" in stage3_res and stage3_res["best_multi_policy_traj"] is not None:
    best_policy_traj = stage3_res["best_multi_policy_traj"]
else:
    # fallback: only for old single-node pipeline
    best_single_node = int(stage3_res["single_df"].iloc[0]["node"])
    best_policy_traj = stage3_res["best_details"][best_single_node]["policy_traj"]

print("Stage 4 uses final policy:")
print(best_policy)
print("Primary policy node:", best_node)

stage4_res = summarize_recovery(
    healthy_traj=stage1_res["healthy_traj"],
    lesion_traj=stage1_res["lesion_traj"],
    policy_traj=stage3_res["best_policy_traj"],
    policy=stage3_res["best_policy"],
    roi_names=roi_names,

    alpha_traj=BEST_CONFIG["alpha_traj"],
    beta_fc=BEST_CONFIG["beta_fc"],
    lambda_node=BEST_CONFIG["lambda_node"],
    lambda_strength=BEST_CONFIG["lambda_strength"],
    lambda_temporal=BEST_CONFIG["lambda_temporal"],
    distance_metric="rmse",
)

print(stage4_res["metrics"])
display(stage4_res["time_recovery_df"].head())
display(stage4_res["roi_recovery_df"].head())
display(stage4_res["distance_df"].head())

In [ ]:
# 可视化
time_recovery_panel = plot_time_recovery(
    stage4_res["time_recovery_df"],
    figsize=(12.5, 7.2),
    save_path="./results/stage4_time_resolved_recovery_combo.png",
)

display(time_recovery_panel["phase_df"])
display(time_recovery_panel["summary_df"])
time_recovery_panel["diagnostics"]

In [ ]:
plot_fc_triplet(
    stage4_res["metrics"]["FC_healthy"],
    stage4_res["metrics"]["FC_lesion"],
    stage4_res["metrics"]["FC_policy"],
    neworder=neworder,
)

In [ ]:
# FC差异图
fig, axes, fc_diff_info = plot_stage4_fc_difference_maps(
    stage4_res["metrics"]["FC_healthy"],
    stage4_res["metrics"]["FC_lesion"],
    stage4_res["metrics"]["FC_policy"],
    neworder=neworder if "neworder" in globals() else None,
    save_path="./results/stage4_fc_difference_maps.png",
)

In [ ]:
# FC recovery map
fig, ax, fc_delta_info = plot_stage4_fc_recovery_delta_map(
    stage4_res["metrics"]["FC_healthy"],
    stage4_res["metrics"]["FC_lesion"],
    stage4_res["metrics"]["FC_policy"],
    neworder=neworder if "neworder" in globals() else None,
    save_path="./results/stage4_fc_recovery_delta_map.png",
)

In [ ]:
# Recovery scatter 图
fig, ax, scatter_summary = plot_stage4_recovery_scatter(
    roi_recovery_df=stage4_res["roi_recovery_df"],
    roi_names=roi_names if "roi_names" in globals() else None,
    lesion_idx=lesion_idx,
    top_k_label=5,
    color_by="roi_recovery",   # 也可以改成 "improvement"
    save_path="./results/stage4_recovery_scatter.png",
)
#挤的话调draw_scatter_callout_labels函数的offset_frac=(0.090, 0.080)
print(scatter_summary)

In [ ]:
ax=plot_top_roi_recovery(stage4_res["roi_recovery_df"], top_k=20)
plt.show()

In [ ]:
plot_three_trajectory_distance(stage4_res["distance_df"])

In [ ]:
# lesion_response_vs_recovery_scatter,是一个 Stage 1 ↔ Stage 4 的桥接图
fig, ax, response_recovery_df, response_recovery_summary = plot_lesion_response_vs_recovery_scatter(
        lesion_response_df=stage1_res["lesion_response_df"],
        roi_recovery_df=stage4_res["roi_recovery_df"],
        roi_names=roi_names if "roi_names" in globals() else None,
        lesion_idx=lesion_idx,
        response_col="response_abs",
        recovery_col="roi_recovery",
        label_top_k=5,
        color_by="improvement",   # 也可以换成 "roi_recovery" 或 None
        save_path="./results/stage4_lesion_response_vs_recovery_scatter.png",
    )

print(response_recovery_summary)
response_recovery_df.head()

In [ ]:
fig_node, ax_node, fig_edge, ax_edge, causal_graph_info = plot_stage4_directed_causal_intervention_graph_split(
        core_edge_df=lesion_core_edges,
        stage4_metrics=stage4_res["metrics"],
        lesion_idx=lesion_idx,
        best_policy=best_policy,
        roi_recovery_df=stage4_res["roi_recovery_df"],
        stage1_response_df=stage1_res["lesion_response_df"],
        roi_names=roi_names if "roi_names" in globals() else None,
        top_k_edges=8,
        mode="lesion_neighborhood",
        show_policy_to_lesion_link=True,
        save_prefix="./results/stage4_directed_causal_graph",
    )

print(causal_graph_info["path_info"])
display(causal_graph_info["focus_edges"].head(20))
display(causal_graph_info["node_df"].head())

In [ ]:
best_policy = stage3_res["best_policy"]
best_node = int(best_policy[0]["node"])

print("best_node =", best_node)
print("lesion_idx =", lesion_idx)
print("best_policy =", best_policy)

path_check = extract_policy_lesion_paths(
    lesion_core_edges=lesion_core_edges,   # 注意：用完整 lesion_core_edges，不是 direct_lesion_edges
    lesion_idx=lesion_idx,
    best_policy=best_policy,
    direction="auto",
    max_hops=5,
    top_k_paths=5,
    score_col="abs_val",
    score_mode="mean",
    path_mode="topk_union",
    allow_undirected_fallback=True,
)

print("has_path:", path_check["has_path"])
print("direction_used:", path_check["direction_used"])
print("search_type:", path_check["search_type"])
print("selected_paths:", path_check["selected_paths"])
print("selected_path_scores:", path_check["selected_path_scores"])

In [ ]:
# 画 shortest path 版本
fig_union, ax_union, path_info_short = plot_stage4_policy_lesion_path_graph(
    lesion_core_edges=lesion_core_edges,
    lesion_idx=lesion_idx,
    best_policy=best_policy,
    roi_names=roi_names if "roi_names" in globals() else None,

    direction="auto",
    max_hops=5,
    top_k_paths=1,
    path_mode="shortest",

    score_col="abs_val",
    score_mode="mean",
    allow_undirected_fallback=True,

    figsize=(12.5, 6.2),
    show_path_summary=True,
    save_path="./results/stage4_policy_lesion_shortest_path_graph.png",
)

print(path_info_short["path_res"]["selected_paths"])
display(path_info_short["path_edge_df"])

In [ ]:
# 画 Top-K paths union 版本:干预节点到病灶节点之间所有可能经过的节点
fig_union, ax_union, path_info_union = plot_stage4_policy_lesion_path_graph(
    lesion_core_edges=lesion_core_edges,
    lesion_idx=lesion_idx,
    best_policy=best_policy,
    roi_names=roi_names if "roi_names" in globals() else None,

    direction="auto",
    max_hops=5,
    top_k_paths=5,
    path_mode="topk_union",

    score_col="abs_val",
    score_mode="mean",
    allow_undirected_fallback=True,

    figsize=(12.5, 6.2),
    show_path_summary=True,
    save_path="./results/stage4_policy_lesion_topk_union_path_graph.png",
)

print(path_info_union["path_res"]["selected_paths"])
print(path_info_union["path_res"]["selected_path_scores"])
display(path_info_union["path_edge_df"])

In [ ]:
best_policy = stage3_res["best_policy"]
best_node = int(best_policy[0]["node"])

recovery_path_pair = plot_stage4_policy_lesion_recovery_path_graph_pair(
    core_edge_df=lesion_core_edges,
    stage4_metrics=stage4_res["metrics"],
    best_policy=best_policy,
    lesion_idx=lesion_idx,
    roi_names=roi_names if "roi_names" in globals() else None,

    direction="auto",
    top_k_paths=4,
    max_hops=5,

    save_prefix="./results/stage4_policy_lesion_recovery_path",
)

print("Shortest path:")
print(recovery_path_pair["shortest"]["info"]["selected_paths"])
display(recovery_path_pair["shortest"]["info"]["selected_edge_df"])

print("Top-K union paths:")
print(recovery_path_pair["topk_union"]["info"]["selected_paths"])
display(recovery_path_pair["topk_union"]["info"]["selected_edge_df"])